# 01 기본 RAG의 원리와 문서 검색 — 문서가 어떻게 검색되는가

반도체 공정 설명 한 쪽으로 RAG의 **문서 준비와 검색**을 처음부터 따라갑니다(45분, 주피터만 사용).

`PDF 한 쪽 읽기 → 원문 확인 → 청킹 → 오버랩 비교 → 임베딩 → Chroma 색인 → 질문 검색 → 추가 사례(자막·그림 캡션)`

- 문서: 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) **책자 6쪽 = PDF 8번째 페이지** ‘2. 반도체 Value Chain Overview (8대 공정)’ · [발행처 원문(8번째 페이지)](https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf#page=8)
- 원본 PDF는 함께 배포하지 않습니다. 노트북 옆 `data/samilpwc_semicon-guidebook.pdf`가 없으면 발행처 주소에서 내려받습니다. 내려받을 수 없는 환경이면 위 링크에서 직접 받아 그 위치에 두세요.
- 앞 챕터를 실행하지 않아도 됩니다. 새 커널에서 첫 셀부터 차례로 실행합니다.
- 이 노트북은 임베딩 모델(text-embedding-3-small)만 씁니다. 답변 생성은 02에서 합니다.

In [1]:
# 현재 Python 3.12 커널에 실습 패키지를 설치합니다(처음 한 번).
# 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행하세요.
%pip install langchain-core==1.8.4 langchain-openai==1.8.5 langchain-chroma==1.2.0 chromadb==1.5.9 openai==3.59.0 pypdf==7.1.1

  Using cached openai-3.59.0-py3-none-any.whl.metadata (45 kB)
  Using cached certifi-2026.11.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached jsonpatch-1.48-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.22-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.16.2-py3-none-any.whl.metadata (40 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (42 kB)
  Using cached httpx2-2.16.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached jiter-0.19.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 

  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached jsonpointer-3.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached charset_normalizer-3.5.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (43 kB)
  Using cached regex-2026.11.25-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 41.1 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 107.9 MB/s  0:00:00m0:00:01
Using cached openai-3.59.0-py3-none-any.whl (2.5 MB)
Using cached httpx2-2.16.2-py3-none-any.whl (97 kB)
Using cached httpcore2-2.16.2-py3-none-any.whl (85 kB)
Using cached jiter-0.19.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (338 kB)
Using cached langsmith-0.16.2-py3-none-any.whl (1.0 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached pydantic-2.14.4-py3-none-any.whl (494 kB)
Using cached pydantic_core-2.50.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (2.3 MB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cached uuid_utils-0.19.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (330 kB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached certifi-2026.11.20-py3-none-any.whl (127 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 99.4 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37

  Attempting uninstall: typing-extensions━━━━━━━━━━━━━━━━━━━━━━━━━  7/90 [urllib3]ts]
    Found existing installation: typing_extensions 4.15.0━━━━━  7/90 [urllib3]
    Uninstalling typing_extensions-4.15.0:━━━━━━━━━━━━━━━━━━━━  7/90 [urllib3]
      Successfully uninstalled typing_extensions-4.15.0━━━━━━━━━━━  8/90 [typing-extensions]
  Attempting uninstall: jsonpointer0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/90 [numpy]]f]ions]
    Found existing installation: jsonpointer 3.1.1━━━━━━━━━━━━ 24/90 [numpy]
    Uninstalling jsonpointer-3.1.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/90 [numpy]
      Successfully uninstalled jsonpointer-3.1.1━━━━━━━━━━━━━━━━━━ 28/90 [jsonpointer]
  Attempting uninstall: idna0m━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28/90 [jsonpointer]
    Found existing installation: idna 3.11━━━━━━━━━━━━━━━━━━━━ 28/90 [jsonpointer]
    Uninstalling idna-3.11:━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28/90 [jsonpointer]
      Successfully uninstalled idna-3.11━━━━━━━━━━━━━━━━━━━━━━ 28/90 [jsonpointer]
   ━━━━━━━━━━━

In [2]:
# ▶ 설정 셀 — 따옴표 안의 값만 바꾸세요. 모델명은 확정값이라 바꾸지 않습니다.
# 키가 든 이 셀은 화면 공유·캡처하거나 AI 도우미에 붙여넣지 않습니다. 파일을 공유할 때는 자리표시 문자열로 되돌립니다.
API_KEY = ""
EMBEDDING_BASE_URL = ""    # 안내받은 임베딩 모델 주소(https://..., 채팅 주소와 다름)
EMBEDDING_MODEL_NAME = "text-embedding-3-small"        # 문서·질문 임베딩 모델(확정값)

In [3]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
NEEDED = ["API_KEY", "EMBEDDING_BASE_URL"]
missing = [name for name in NEEDED if globals()[name].startswith("여기에") or not globals()[name].strip()]
if missing:   # 임베딩 주소가 없을 때 채팅 주소로 대신하지 않는다
    raise ValueError(", ".join(missing) + "에 안내받은 값을 설정 셀의 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, api_key=API_KEY, base_url=EMBEDDING_BASE_URL,
                              check_embedding_ctx_length=False)
print("임베딩 모델:", EMBEDDING_MODEL_NAME, "· 벡터 길이:", len(embeddings.embed_query("시험")))

임베딩 모델: text-embedding-3-small · 벡터 길이: 1536


## 1. PDF 준비와 한 쪽 읽기

책자에 인쇄된 쪽 번호(6)와 PDF 뷰어의 페이지 번호(8)는 다릅니다. 파이썬은 0부터 세므로 8번째 페이지는 `pages[7]`입니다.

In [4]:
import os, urllib.request
from pathlib import Path

PDF_URL = "https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf"
PDF_FILE = Path("data/samilpwc_semicon-guidebook.pdf")       # 원본은 함께 배포하지 않음 → 없으면 내려받음
SOURCE = "삼일PwC 『반도체 산업 Business Guidebook』(2025.12)"
PAGE_INDEX = 7                       # 파이썬은 0부터 셈 → PDF 뷰어의 8번째 페이지
PDF_PAGE, BOOK_PAGE = PAGE_INDEX + 1, 6   # 책자에 인쇄된 쪽 번호는 6
VIEW_URL = f"{PDF_URL}#page={PDF_PAGE}"   # #page=8은 뷰어 이동용. 코드에서는 PAGE_INDEX로 페이지를 고른다

def ensure_pdf():
    if not PDF_FILE.is_file():
        PDF_FILE.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(PDF_URL, timeout=60) as r:     # 기본 요청으로 받는다(브라우저 흉내 헤더는 거절됨)
                PDF_FILE.write_bytes(r.read())
            print("발행처에서 내려받음:", PDF_URL)
        except Exception as e:
            raise RuntimeError(f"PDF를 받지 못했습니다({type(e).__name__}). 브라우저로 {PDF_URL} 을 내려받아 "
                               f"이 노트북 옆 {PDF_FILE} 에 두고 다시 실행하세요.")
    return PDF_FILE

from pypdf import PdfReader
reader = PdfReader(str(ensure_pdf()))
page = reader.pages[PAGE_INDEX]
raw = page.extract_text()
print(f"PDF 전체 {len(reader.pages)}쪽 중 {PDF_PAGE}번째 페이지(책자 {BOOK_PAGE}쪽) · 추출 글자 수 {len(raw)}")
print(raw)

발행처에서 내려받음: https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf
PDF 전체 96쪽 중 8번째 페이지(책자 6쪽) · 추출 글자 수 1059
웨이퍼 제조의 경우, 실리콘 잉곳(Ingot)의 제조와 이를 
절단, 연마, 세척하여, 최종 검사를 통해 웨이퍼를 생산하
는 공정을 의미한다. 잉곳은 웨이퍼로 제조되기 전의 실
리콘 기둥으로, 이를 절단하여 반도체의 기초 제조인 웨
이퍼를 생산한다. 이러한 웨이퍼 제조 공정을 통해 생산
된 웨이퍼는 산화-포토-식각 공정을 거쳐 회로가 주입
된다. 산화공정은 웨이퍼 표면에 산화막을 형성하는 과
정으로, 산화막은 이후 과정인 식각 및 이온 주입 과정
에서 방지막 역할을 수행하여, 웨이퍼가 설계된 회로대
로 제조될 수 있도록 한다. 포토공정은 웨이퍼에 회로를 
그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방
식으로 진행된다. 식각 공정에서는 포토공정에서 입혀
진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 
및 이온주입공정을 통해 전기적 특성을 부여하게 된다 
(8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과
정에서 발생하는 불순물을 제거하기 위한 세정 공정을 
거친다.) 
금속배선 공정을 통해 배선까지 완료된 반도체에 테스트 
및 패키징 공정이 진행되며 이러한 공정은 통상 후공정
으로 지칭된다. 테스트 공정에서는 웨이퍼 내의 개별 Die 
품질을 테스트하게 되며, 이중 양품을 절단하여 칩으로 
제조한다. 절단된 칩은 패키징 과정을 거치게 되며, 패키
징 테스트 이후 최종 제품으로 판매된다.
감광제
산화막
웨이퍼
감광제
산화막 제 거 제 거
웨이퍼
식각 전 웨이퍼 식각 후 웨이퍼
잉곳 웨이퍼
절단
웨이퍼 제조 전공정 후공정
웨이퍼 제조 산화공정 포토공정 식각공정
패키징 공정 테스트 공정 금속배선 공정
웨이퍼
Partial edge die
Good die
Defective die
Defect
박막 박막금속


## 2. 원문 확인 — 본문과 그림 속 글자 구분하기

위 출력의 뒷부분에는 공정 그림 안의 짧은 이름표(‘감광제’, ‘웨이퍼’ …)가 섞여 있습니다. 글자 크기와 위치로 **제목(20pt)과 본문(10pt)** 만 남깁니다.

In [5]:
from pypdf import PdfReader

def page_runs(page):
    """페이지의 글자 묶음마다 (글자 크기, 세로 위치, 글자)를 모은다."""
    runs = []
    page.extract_text(visitor_text=lambda t, cm, tm, fd, fs: runs.append((round(fs * (tm[0] or 1), 1), tm[5], t)))
    return runs

def read_body(page):
    """제목(20pt)과 본문(10pt, 공정 그림 아래쪽)만 남기고 그림 속 짧은 이름표는 뺀다."""
    runs = page_runs(page)
    title = "".join(t for size, y, t in runs if size >= 19).replace("\n", "").strip()
    body = "".join(t for size, y, t in runs if 9.5 <= size <= 10.5 and y < 360).replace("\n", "").strip()
    return title, body

from collections import Counter
print("글자 크기별 묶음 수:", dict(Counter(size for size, y, t in page_runs(page))))
TITLE, BODY = read_body(page)
print("제목:", TITLE)
print(f"본문 {len(BODY)}자:")
print(BODY)

글자 크기별 묶음 수: {12.0: 4, 10.0: 51, 1.0: 20, 7.0: 26, 5.0: 4, 8.0: 38, 6.6: 27, 6.9: 6, 9.0: 1, 6.8: 7, 6.0: 5, 20.0: 9, 19.6: 1}
제목: 2. 반도체 Value Chain Overview (8대 공정)
본문 663자:
웨이퍼 제조의 경우, 실리콘 잉곳(Ingot)의 제조와 이를 절단, 연마, 세척하여, 최종 검사를 통해 웨이퍼를 생산하는 공정을 의미한다. 잉곳은 웨이퍼로 제조되기 전의 실리콘 기둥으로, 이를 절단하여 반도체의 기초 제조인 웨이퍼를 생산한다. 이러한 웨이퍼 제조 공정을 통해 생산된 웨이퍼는 산화-포토-식각 공정을 거쳐 회로가 주입된다. 산화공정은 웨이퍼 표면에 산화막을 형성하는 과정으로, 산화막은 이후 과정인 식각 및 이온 주입 과정에서 방지막 역할을 수행하여, 웨이퍼가 설계된 회로대로 제조될 수 있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다. 식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다 (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정을 거친다.) 금속배선 공정을 통해 배선까지 완료된 반도체에 테스트 및 패키징 공정이 진행되며 이러한 공정은 통상 후공정으로 지칭된다. 테스트 공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하게 되며, 이중 양품을 절단하여 칩으로 제조한다. 절단된 칩은 패키징 과정을 거치게 되며, 패키징 테스트 이후 최종 제품으로 판매된다.


RAG에서 확인할 두 문장의 위치(본문 몇 번째 글자부터인지)를 먼저 찾아 둡니다.

In [6]:
PHOTO = "포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다."
ETCH = "식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다"
for name, sentence in [("포토 문장", PHOTO), ("식각 문장", ETCH)]:
    at = BODY.find(sentence)
    print(f"{name}: 본문 {at}~{at + len(sentence)}자")

포토 문장: 본문 289~343자
식각 문장: 본문 344~420자


## 3. 청킹 — 일정한 글자 수로 자르기

200자마다 자릅니다. 문장 중간에서도 자르므로 한 문장이 두 조각에 나뉠 수 있습니다.

In [7]:
from langchain_core.documents import Document

def split_text(text, size, overlap):
    """size 글자마다 자르고, 다음 조각은 앞 조각의 끝 overlap 글자부터 시작한다 → (시작, 끝) 목록"""
    assert 0 <= overlap < size, "겹침은 0 이상, 조각 크기보다 작아야 합니다"
    spans, start = [], 0
    while True:
        end = min(start + size, len(text))
        spans.append((start, end))
        if end >= len(text):
            return spans
        start += size - overlap

def make_chunks(title, body, size, overlap):
    return [Document(page_content=body[s:e], metadata={
                "chunk_id": f"pwc-p{PDF_PAGE}-c{i:02d}", "source": SOURCE, "title": title, "file": PDF_FILE.name,
                "book_page": BOOK_PAGE, "pdf_page": PDF_PAGE, "url": VIEW_URL, "start": s, "end": e,
                "chunk_size": size, "overlap": overlap})
            for i, (s, e) in enumerate(split_text(body, size, overlap), start=1)]

def show_chunks(chunks):
    for c in chunks:
        m = c.metadata
        print(f"[{m['chunk_id']}] 본문 {m['start']}~{m['end']}자 ({len(c.page_content)}자) | {c.page_content}")

chunks0 = make_chunks(TITLE, BODY, size=200, overlap=0)
show_chunks(chunks0)

[pwc-p8-c01] 본문 0~200자 (200자) | 웨이퍼 제조의 경우, 실리콘 잉곳(Ingot)의 제조와 이를 절단, 연마, 세척하여, 최종 검사를 통해 웨이퍼를 생산하는 공정을 의미한다. 잉곳은 웨이퍼로 제조되기 전의 실리콘 기둥으로, 이를 절단하여 반도체의 기초 제조인 웨이퍼를 생산한다. 이러한 웨이퍼 제조 공정을 통해 생산된 웨이퍼는 산화-포토-식각 공정을 거쳐 회로가 주입된다. 산화공정은 웨이퍼 표
[pwc-p8-c02] 본문 200~400자 (200자) | 면에 산화막을 형성하는 과정으로, 산화막은 이후 과정인 식각 및 이온 주입 과정에서 방지막 역할을 수행하여, 웨이퍼가 설계된 회로대로 제조될 수 있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다. 식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정
[pwc-p8-c03] 본문 400~600자 (200자) | 을 통해 전기적 특성을 부여하게 된다 (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정을 거친다.) 금속배선 공정을 통해 배선까지 완료된 반도체에 테스트 및 패키징 공정이 진행되며 이러한 공정은 통상 후공정으로 지칭된다. 테스트 공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하게 되며, 이중 양품을
[pwc-p8-c04] 본문 600~663자 (63자) |  절단하여 칩으로 제조한다. 절단된 칩은 패키징 과정을 거치게 되며, 패키징 테스트 이후 최종 제품으로 판매된다.


## 4. 오버랩 비교 — 겹침 0자 vs 60자

같은 200자 조각에 **겹침 60자**를 줍니다. 다음 조각은 앞 조각의 마지막 60자에서 시작합니다. 이웃한 두 조각에서 실제로 겹친 글자를 출력하고, 식각 문장이 한 조각 안에 온전히 들어 있는지 비교합니다.

In [8]:
chunks60 = make_chunks(TITLE, BODY, size=200, overlap=60)
show_chunks(chunks60)

def overlap_of(a, b):
    s, e = b.metadata["start"], a.metadata["end"]
    return BODY[s:e] if s < e else ""

print("\n이웃 조각의 실제 겹침")
for chunks, label in [(chunks0, "겹침 0"), (chunks60, "겹침 60")]:
    a, b = chunks[1], chunks[2]
    same = overlap_of(a, b)
    print(f"  {label}: [{a.metadata['chunk_id']}] 끝 ↔ [{b.metadata['chunk_id']}] 앞 · 겹친 글자 {len(same)}자 | {same or '(겹침 없음)'}")
    assert a.page_content.endswith(same) and b.page_content.startswith(same)

print("\n식각 문장이 통째로 들어 있는 조각")
for chunks, label in [(chunks0, "겹침 0"), (chunks60, "겹침 60")]:
    whole = [c.metadata["chunk_id"] for c in chunks if ETCH in c.page_content]
    both = [c.metadata["chunk_id"] for c in chunks if ETCH in c.page_content and PHOTO in c.page_content]
    print(f"  {label}: {whole or '없음 — 문장이 두 조각에 나뉨'} · 포토·식각 문장을 함께 담은 조각 {both or '없음'}")

[pwc-p8-c01] 본문 0~200자 (200자) | 웨이퍼 제조의 경우, 실리콘 잉곳(Ingot)의 제조와 이를 절단, 연마, 세척하여, 최종 검사를 통해 웨이퍼를 생산하는 공정을 의미한다. 잉곳은 웨이퍼로 제조되기 전의 실리콘 기둥으로, 이를 절단하여 반도체의 기초 제조인 웨이퍼를 생산한다. 이러한 웨이퍼 제조 공정을 통해 생산된 웨이퍼는 산화-포토-식각 공정을 거쳐 회로가 주입된다. 산화공정은 웨이퍼 표
[pwc-p8-c02] 본문 140~340자 (200자) | 웨이퍼 제조 공정을 통해 생산된 웨이퍼는 산화-포토-식각 공정을 거쳐 회로가 주입된다. 산화공정은 웨이퍼 표면에 산화막을 형성하는 과정으로, 산화막은 이후 과정인 식각 및 이온 주입 과정에서 방지막 역할을 수행하여, 웨이퍼가 설계된 회로대로 제조될 수 있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행
[pwc-p8-c03] 본문 280~480자 (200자) |  있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다. 식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다 (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정
[pwc-p8-c04] 본문 420~620자 (200자) |  (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정을 거친다.) 금속배선 공정을 통해 배선까지 완료된 반도체에 테스트 및 패키징 공정이 진행되며 이러한 공정은 통상 후공정으로 지칭된다. 테스트 공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하게 되며, 이중 양품을 절단하여 칩으로 제조한다. 절단된 
[pwc-p8-c05] 본문 560~663자 (103자) | 공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하

이 노트북은 이후 **200자·겹침 60자** 조각으로 색인합니다. 조각마다 문서명·책자 쪽·PDF 쪽·조각 ID·본문 위치를 metadata로 붙였습니다.

In [9]:
print(chunks60[2].page_content)
chunks60[2].metadata

 있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다. 식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다 (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정


{'chunk_id': 'pwc-p8-c03',
 'source': '삼일PwC 『반도체 산업 Business Guidebook』(2025.12)',
 'title': '2. 반도체 Value Chain Overview (8대 공정)',
 'file': 'samilpwc_semicon-guidebook.pdf',
 'book_page': 6,
 'pdf_page': 8,
 'url': 'https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf#page=8',
 'start': 280,
 'end': 480,
 'chunk_size': 200,
 'overlap': 60}

## 5. 임베딩 — 조각 하나를 숫자 목록으로

문서 조각과 질문을 **같은** 임베딩 모델로 바꿔야 비교할 수 있습니다.

In [10]:
vec = embeddings.embed_documents([chunks60[2].page_content])[0]
print(f"[{chunks60[2].metadata['chunk_id']}] → 벡터 길이 {len(vec)} · 앞 5개 값 {[round(v, 4) for v in vec[:5]]}")

[pwc-p8-c03] → 벡터 길이 1536 · 앞 5개 값 [-0.0324, 0.0027, 0.0158, 0.0445, -0.0139]


## 6. Chroma 색인

조각 원문·벡터·metadata를 로컬 폴더 `chroma_db/`에 저장합니다. 조각 ID를 저장 ID로 쓰므로, 이 셀을 **다시 실행해도 새로 추가 0개**입니다. 조각 크기나 겹침을 바꾸면 다른 컬렉션 이름이 됩니다.

In [11]:
import hashlib
from langchain_chroma import Chroma

def open_index(chunks):
    """설정(조각 크기·겹침)·본문·임베딩 모델이 같으면 같은 컬렉션 이름 → 다시 실행해도 중복 저장하지 않는다."""
    m = chunks[0].metadata
    key = EMBEDDING_MODEL_NAME + "".join(c.page_content for c in chunks)
    name = f"pwc-p{PDF_PAGE}-{m['chunk_size']}-{m['overlap']}-{hashlib.sha256(key.encode('utf-8')).hexdigest()[:8]}"
    store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                   collection_metadata={"hnsw:space": "cosine"})            # 거리 = 코사인 거리(작을수록 가까움)
    ids = [c.metadata["chunk_id"] for c in chunks]
    have = set(store.get(ids=ids)["ids"])
    new = [c for c in chunks if c.metadata["chunk_id"] not in have]
    if new:
        store.add_documents(new, ids=[c.metadata["chunk_id"] for c in new])
    print(f"컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 조각 {len(store.get()['ids'])}개")
    return store

def search(store, question, k=3, **kw):
    found = store.similarity_search_with_score(question, k=k, **kw)
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_hits(hits):
    for h in hits:
        print(f"{h['rank']}위 [{h['chunk_id']}] 거리 {h['distance']} · {h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
              f" · 본문 {h['start']}~{h['end']}자")
        print("    " + h["text"])

store = open_index(chunks60)
store_again = open_index(chunks60)       # 같은 설정으로 한 번 더 → 새로 추가 0개

컬렉션 pwc-p8-200-60-074c96ec · 새로 추가 5개 · 저장된 조각 5개
컬렉션 pwc-p8-200-60-074c96ec · 새로 추가 0개 · 저장된 조각 5개


## 7. 질문으로 검색하고 원문으로 돌아가기

질문을 같은 임베딩 모델로 바꿔 코사인 거리가 가까운 조각 3개를 찾습니다. 각 결과의 metadata로 **어느 문서의 몇 쪽, 본문 몇 번째 글자**에서 왔는지 확인합니다.

In [12]:
QUESTION = "감광액이 덮이지 않은 부분을 없애는 단계는?"
hits = search(store, QUESTION, k=3)
show_hits(hits)

1위 [pwc-p8-c03] 거리 0.5723 · 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) 책자 6쪽 = PDF 8번째 페이지 · 본문 280~480자
     있도록 한다. 포토공정은 웨이퍼에 회로를 그려넣는 과정으로, 감광액 도포 후 노광, 현상하는 방식으로 진행된다. 식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다 (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정
2위 [pwc-p8-c04] 거리 0.7243 · 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) 책자 6쪽 = PDF 8번째 페이지 · 본문 420~620자
     (8대 공정에 포함되어 있지는 않지만, 산화-포토-식각 과정에서 발생하는 불순물을 제거하기 위한 세정 공정을 거친다.) 금속배선 공정을 통해 배선까지 완료된 반도체에 테스트 및 패키징 공정이 진행되며 이러한 공정은 통상 후공정으로 지칭된다. 테스트 공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하게 되며, 이중 양품을 절단하여 칩으로 제조한다. 절단된 
3위 [pwc-p8-c05] 거리 0.7763 · 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) 책자 6쪽 = PDF 8번째 페이지 · 본문 560~663자
    공정에서는 웨이퍼 내의 개별 Die 품질을 테스트하게 되며, 이중 양품을 절단하여 칩으로 제조한다. 절단된 칩은 패키징 과정을 거치게 되며, 패키징 테스트 이후 최종 제품으로 판매된다.


In [13]:
from IPython.display import HTML, display
top = hits[0]
print("1위 조각이 원문과 같은가:", BODY[top["start"]:top["end"]] == top["text"], "· 식각 문장 포함:", ETCH in top["text"])
display(HTML(f'원문 확인: <a href="{top["url"]}" target="_blank">{top["source"]} 책자 {top["book_page"]}쪽 = PDF {top["pdf_page"]}번째 페이지 열기</a>'))

1위 조각이 원문과 같은가: True · 식각 문장 포함: True


## 8. 추가 사례 — 영상 자막과 그림 캡션도 Document가 된다

PDF가 아닌 자료도 **검색할 글자**(page_content)와 **돌아갈 주소**(metadata)로 나누면 같은 방식으로 색인할 수 있습니다.

영상: [MIT OCW 6.774 Lecture 20 Etching – Introduction](https://www.youtube.com/watch?v=4q5k1hLfyrM) (자막: 같은 영상을 임베드한 [MIT 공식 WebVTT](https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt), 영상 설명의 이용 조건 CC BY-NC-SA — 기업 교육에서 자막을 재사용할 때는 허용 범위를 확인) · 그림: [삼성반도체 「반도체 8대 공정 5탄 ‘식각 공정’」](https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/) 본문 그림의 캡션과 대체 텍스트. 임베딩에는 **글자(자막·캡션·대체 텍스트)만** 넣습니다. 이미지 자체를 분석하거나 검색하지 않습니다.

자료를 가져오지 못하면 상태와 원문 링크만 표시하고 넘어갑니다. 위의 PDF 실습에는 영향이 없습니다.

In [14]:
import html, re, urllib.request
from langchain_core.documents import Document

VIDEO_ID = "4q5k1hLfyrM"
VIDEO_TITLE = "MIT OCW 6.774 Lecture 20: Etching – Introduction (Judy Hoyt)"
VIDEO_PAGE = f"https://www.youtube.com/watch?v={VIDEO_ID}"
MIT_VTT = ("https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/"
           "1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt")          # 같은 영상을 임베드한 MIT 공식 페이지의 영어 자막
ARTICLE_URL = ("https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/"
               "eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/")
ARTICLE_TITLE = "삼성반도체 「반도체 8대 공정 5탄, 반도체 회로패턴의 완성 ‘식각 공정’」"

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode("utf-8", errors="replace")

def seconds(t):                       # "00:02:00.370" → 120.37
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

def caption_text():
    """유튜브 자막을 먼저 요청하고, 비어 있거나 실패하면 MIT 공식 WebVTT를 쓴다. (자막 원문, 자막 출처)"""
    try:
        body = fetch(f"https://www.youtube.com/api/timedtext?v={VIDEO_ID}&lang=en&fmt=vtt")
        if "-->" in body:
            return body, "YouTube 자막"
        print("유튜브 직접 자막 조회: 빈 응답 → MIT 공식 WebVTT 사용")
    except Exception as e:
        print(f"유튜브 직접 자막 조회 실패({type(e).__name__}) → MIT 공식 WebVTT 사용")
    return fetch(MIT_VTT), MIT_VTT

def load_video_docs(minutes=15, window=30):
    """앞 minutes분 자막을 window초 묶음으로 → Document(본문=자막, metadata=시작 시각·재생 링크·자막 출처)"""
    try:
        vtt, caption_source = caption_text()
    except Exception as e:
        print(f"자막을 가져오지 못했습니다({type(e).__name__}). 영상 원문: {VIDEO_PAGE}")
        return []
    groups = {}
    for start, text in re.findall(r"(\d\d:\d\d:\d\d\.\d{3}) --> [^\n]*\n(.*?)(?:\n\n|\Z)", vtt, flags=re.S):
        sec = seconds(start)
        text = " ".join(text.split())
        if text and sec < minutes * 60:
            groups.setdefault(int(sec // window) * window, []).append(text)
    return [Document(page_content=" ".join(lines), metadata={
                "media": "video", "doc_id": f"video-{t:04d}s", "start_sec": t, "time": f"{t // 60:02d}:{t % 60:02d}",
                "url": f"{VIDEO_PAGE}&t={t}s", "thumbnail": f"https://i.ytimg.com/vi/{VIDEO_ID}/hqdefault.jpg",
                "title": VIDEO_TITLE, "caption_source": caption_source})
            for t, lines in sorted(groups.items())]

def load_figure_docs():
    """기사 본문 그림의 실제 캡션(▲ …)과 대체 텍스트 → Document(본문=캡션·대체 텍스트, metadata=이미지 URL·기사 출처)"""
    try:
        page = fetch(ARTICLE_URL)
    except Exception as e:
        print(f"기사를 가져오지 못했습니다({type(e).__name__}). 기사 원문: {ARTICLE_URL}")
        return []
    docs, seen = [], set()
    for m in re.finditer(r'<img[^>]+src="([^"]+etching-a-circuit-pattern_\d+_pc[^"]*)"[^>]*alt="([^"]*)"', page):
        src, alt = html.unescape(m.group(1)), html.unescape(m.group(2)).strip()
        after = re.sub(r"<[^>]+>", " ", page[m.end():m.end() + 1500])
        cap = re.search(r"▲\s*([^▲<]{2,60}?)\s{2,}", html.unescape(after))
        if src in seen or not cap:            # 같은 그림의 반응형 중복과 캡션 없는 머리 그림은 건너뜀
            continue
        seen.add(src)
        caption = "▲ " + cap.group(1).strip()
        docs.append(Document(page_content=f"그림 캡션: {caption} / 대체 텍스트: {alt}", metadata={
            "media": "figure", "doc_id": f"figure-{len(docs) + 1}", "caption": caption, "alt": alt,
            "image_url": src, "url": ARTICLE_URL, "title": ARTICLE_TITLE}))
    return docs

video_docs = load_video_docs()
figure_docs = load_figure_docs()
print(f"영상 자막 Document {len(video_docs)}개(앞 15분, 30초 묶음) · 그림 Document {len(figure_docs)}개")

유튜브 직접 자막 조회: 빈 응답 → MIT 공식 WebVTT 사용
영상 자막 Document 30개(앞 15분, 30초 묶음) · 그림 Document 2개


In [15]:
# 무엇을 검색하고(page_content) 어디로 돌아가는지(metadata)
for d in [next((v for v in video_docs if v.metadata["start_sec"] == 120), None), *figure_docs]:
    if d is None:
        continue
    print("page_content:", d.page_content)
    print("metadata    :", d.metadata)
    print()

page_content: I only have two or three slides on wet etching. Wet etching is still used in research and development, but it's very rarely used in a manufacturing process, and we'll see a little bit why that is. So the next two lectures, what I want to try to do is give you a basic introduction to etching, mostly plasma etching, some specific examples. The next lecture, I'm going to have some specific emphasis on gate etching, polysilicon gate etching, since that's really very important to the CMOS process. And we'll talk a little bit, next lecture as well,
metadata    : {'media': 'video', 'doc_id': 'video-0120s', 'start_sec': 120, 'time': '02:00', 'url': 'https://www.youtube.com/watch?v=4q5k1hLfyrM&t=120s', 'thumbnail': 'https://i.ytimg.com/vi/4q5k1hLfyrM/hqdefault.jpg', 'title': 'MIT OCW 6.774 Lecture 20: Etching – Introduction (Judy Hoyt)', 'caption_source': 'https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd

## 정리

| 단계 | 이 노트북에서 확인한 것 |
|---|---|
| 원문 | PDF 8번째 페이지 = 책자 6쪽. 그림 속 이름표를 빼고 본문만 사용 |
| 청킹 | 겹침 0에서는 식각 문장이 두 조각에 나뉘고, 겹침 60에서는 한 조각에 온전히 들어감 |
| 색인 | 조각 ID가 같으면 다시 저장하지 않음(새로 추가 0개) |
| 검색 | 반환된 조각의 metadata로 문서·쪽·본문 위치를 되짚어 원문과 대조 |
| 추가 사례 | 자막 → 재생 시각 링크, 그림 캡션·대체 텍스트 → 이미지 URL·기사 주소를 metadata로 보존 |

02에서는 검색된 원문을 답변 모델에 넘겨 **답과 출처**를 만듭니다.